## 2.2 HumanInTheLoopMiddleware中间件

HumanInTheLoopMiddleware（人在环中间件、人工审核中间件）在 `工具调用前` 中断Agent运行，等待用户对工具调用请求决策。可选的决策有： `approve（同意执行）` 、 `edit（编辑调用配置后执行）` 、 `reject（拒绝执行）` 、 `respond（不执行工具，由人直接给出工具结果）` 。

### 2.2.1 参数说明

```python
HumanInTheLoopMiddleware(
  self,
  interrupt_on: dict[str, bool | InterruptOnConfig],
  *,
  description_prefix: str = 'Tool execution requires approval',
  edit_notice: str | None = _EDIT_NOTICE
)
```

1. 参数1：interrupt_on — 工具名和中断策略的映射

参数类型: ` dict[str, bool | InterruptOnConfig]`,必须为字典，键必须为字符串，而值可以是 bool 或 InterruptOnConfig TypedDict对象(必填键: allowed_decisions；可选键: description、args_schema、when)

策略可以是True、False或InterruptOnConfig对象，精细控制决策选项。

比如:

```python
interrupt_on={
    "get_weather": True,
    "read_email_tool": False,
    "send_email_tool": {
        "allowed_decisions": 
            ["approve", "reject"],
    },
}
```

* True表示所有决策(approve, edit, reject, respond) 都可以选择，相当于:
```python
"get_weather": {
    "allowed_decisions": ["approve", "edit", "reject", "respond"],
}
```

* False表示不中断，即无需审批即可执行。
* InterruptOnConfig 是一个TypedDict的子类，可以用字典直接赋值。支持的Key有：
    * allowed_decisions （必填）：精细控制中断后允许的决策，必须是非空列表，否则创建中间件时会报 ValueError。
    * description ：特定工具的中断描述信息，优先级高于description_prefix，后者会更改 所有 工具中断的描述。也可以传入一个函数 `(tool_call, state, runtime) -> str`，根据调用内容动态生成描述。
    * args_schema （可选）：允许 edit 时参数的 JSON Schema。当前版本（langchain 1.4.2）只是预留了这个键，中间件不会读取它。
    * when （可选）：一个函数，接收本次工具调用的请求，返回 True 才中断，返回 False 则自动放行。例如只在删除 /etc 下的文件时才要求审批：`"when": lambda req: req.tool_call["args"].get("path", "").startswith("/etc")`

参数2：description_prefix — 自定义中断描述

默认为 "Tool execution requires approval" 。描述的默认格式是“前缀 + 工具名 + 参数”，即 `{description_prefix}\n\nTool: 工具名\nArgs: 参数`；如果某个工具单独设置了 description，就完全使用它，本例 send_email_tool 的描述就只有“发送邮件中断啦”。

参数3：edit_notice — 编辑提示

人工 edit 某个工具调用后，会在该工具的结果前面加上这段说明，告诉模型“实际执行的是人修改过的调用”（2.2.3 的输出中可以看到）。默认是一段英文说明，传 None 则不添加。

### 2.2.2 举例过程1：调用前中断
> 注意：本例需要从中断的位置让Agent继续运行，这就需要用到短期记忆

创建Agent时通过checkpointer参数启用了短期记忆，在调用时通过传递相同的config加载记忆。

In [4]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from pydantic import BaseModel, Field

load_dotenv(override=True, verbose=True, dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

custom_profile = {
    "max_input_tokens": 128_000
}

model = ChatOpenAI(
    model="gpt-5.6-luna",
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ["OPENAI_API_BASE"],
    #profile=custom_profile
)

In [18]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage
from langchain.tools import tool
from langgraph.types import Command
from rich import print as rprint
@tool
def get_weather(city: str, is_forcast: bool = False) -> str:
    """
    查询指定城市天气
    
    Args:
        city: 城市名称
        is_forcast: 是否包含明日天气预报？
    """
    res = f"{city}今天天气不错"
    if is_forcast:
        res += "\n明天下雨"
    return res

@tool
def get_news() -> str:
    """

    查询当日新闻
    """
    return "中方三艘油轮通过霍尔木兹海峡"

@tool
def read_email_tool(email_id: str) -> str:
    """
    
    通过邮件ID读取内容的伪函数
    """
    return f"邮件ID：{email_id}\n是空的"
@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """
    
    发送邮件伪函数
    """
    print(">>> 真的执行发送邮件工具了")
    return f"发送给 {recipient} 的邮件标题是：{subject}，内容：{body}"

agent = create_agent(
    model=model,
    tools=[get_weather, get_news, read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[HumanInTheLoopMiddleware(
        interrupt_on={
            "get_weather": True,
            "get_news": True,
            "read_email_tool": False,
            "send_email_tool": {
                "allowed_decisions": ["approve", "reject"],
                "description": "发送邮件中断啦"
            }
        },
        description_prefix="中断啦!",
    )],
)

config = {"configurable": {"thread_id": "1"}}

# 第一次调用：模型给出工具调用后就会暂停，此时所有工具都还没执行
response = agent.invoke(
    {
        "messages": [
            HumanMessage(content="请帮我查询今天北京的天气"
                "查询今日新闻"
                "查看ID为 'sk2131421' 的邮件内容，"
             "向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'"
                "同时做这四件事")
        ]
    },
    config=config,# 通过config传入线程ID，保证同一线程的中断可以被同一线程的决策处理
)

In [19]:
print("==== 第一次 invoke 返回 ====")
print("========= 原始响应 =========")
rprint(response)
print("========= 美化输出 =========")
for msg in response["messages"]:
    msg.pretty_print()
# 关键：看中断信息
interrupts = response.get("__interrupt__", [])
print("========== interrupts ==========")
rprint(interrupts)
print("==== 逐个打印 interrupt 请求 ====")
action_requests = interrupts[0].value["action_requests"]
for action_request in action_requests:
    rprint(action_request)

==== 第一次 invoke 返回 ====
========= 原始响应 =========


{
    'messages': [
        HumanMessage(
            content="请帮我查询今天北京的天气查询今日新闻查看ID为 'sk2131421' 
的邮件内容，向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'同时做这四件事",
            additional_kwargs={},
            response_metadata={},
            id='8fc6be9d-95d5-4dd3-a53b-b0eb4e960b38'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 137,
                    'prompt_tokens': 199,
                    'total_tokens': 336,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 35,
                        'rejected_prediction_tokens': None,
                        'text_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': None,
                        'video_tokens': 0
                    },
                    'cache_creation_input_tokens': 0
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5.6-luna',
                'system_fingerprint': 'fp_qppnj6fihq',
                'id': 'gen_01M463WNJJT88KZRNEM3BHQ2KP',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a10c3e-5112-72d2-bf67-4c9f717ea847-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '北京', 'is_forcast': False},
                    'id': 'call_UDK2lJdr4DAGkDNY01ZHgBAr',
                    'type': 'tool_call'
                },
                {'name': 'get_news', 'args': {}, 'id': 'call_gN8c7oZkHc3bFHJF0YBFox6Q', 'type': 'tool_call'},
                {
                    'name': 'read_email_tool',
                    'args': {'email_id': 'sk2131421'},
                    'id': 'call_dTJ0PSzr6sSHePM7aV1BctnB',
                    'type': 'tool_call'
                },
                {
                    'name': 'send_email_tool',
                    'args': {'body': '你好啊', 'recipient': '15641685664@qq.com', 'subject': '哈哈哈'},
                    'id': 'call_29WAnJco5oOKLHWrWfnNaP7Y',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 199,
                'output_tokens': 137,
                'total_tokens': 336,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'reasoning': 35}
            }
        )
    ],
    '__interrupt__': [
        Interrupt(
            value={
                'action_requests': [
                    {
                        'name': 'get_weather',
                        'args': {'city': '北京', 'is_forcast': False},
                        'description': "中断啦!\n\nTool: get_weather\nArgs: {'city': '北京', 'is_forcast': False}"
                    },
                    {'name': 'get_news', 'args': {}, 'description': '中断啦!\n\nTool: get_news\nArgs: {}'},
                    {
                        'name': 'send_email_tool',
                        'args': {'body': '你好啊', 'recipient': '15641685664@qq.com', 'subject': '哈哈哈'},
                        'description': '发送邮件中断啦'
                    }
                ],
                'review_configs': [
                    {'action_name': 'get_weather', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']},
                    {'action_name': 'get_news', 'allowed_decisions': ['approve', 'edit', 're

========= 美化输出 =========
================================ Human Message =================================

请帮我查询今天北京的天气查询今日新闻查看ID为 'sk2131421' 的邮件内容，向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'同时做这四件事
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_UDK2lJdr4DAGkDNY01ZHgBAr)
 Call ID: call_UDK2lJdr4DAGkDNY01ZHgBAr
  Args:
    city: 北京
    is_forcast: False
  get_news (call_gN8c7oZkHc3bFHJF0YBFox6Q)
 Call ID: call_gN8c7oZkHc3bFHJF0YBFox6Q
  Args:
  read_email_tool (call_dTJ0PSzr6sSHePM7aV1BctnB)
 Call ID: call_dTJ0PSzr6sSHePM7aV1BctnB
  Args:
    email_id: sk2131421
  send_email_tool (call_29WAnJco5oOKLHWrWfnNaP7Y)
 Call ID: call_29WAnJco5oOKLHWrWfnNaP7Y
  Args:
    body: 你好啊
    recipient: 15641685664@qq.com
    subject: 哈哈哈
========== interrupts ==========


[
    Interrupt(
        value={
            'action_requests': [
                {
                    'name': 'get_weather',
                    'args': {'city': '北京', 'is_forcast': False},
                    'description': "中断啦!\n\nTool: get_weather\nArgs: {'city': '北京', 'is_forcast': False}"
                },
                {'name': 'get_news', 'args': {}, 'description': '中断啦!\n\nTool: get_news\nArgs: {}'},
                {
                    'name': 'send_email_tool',
                    'args': {'body': '你好啊', 'recipient': '15641685664@qq.com', 'subject': '哈哈哈'},
                    'description': '发送邮件中断啦'
                }
            ],
            'review_configs': [
                {'action_name': 'get_weather', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']},
                {'action_name': 'get_news', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']},
                {'action_name': 'send_email_tool', 'allowed_decisions': ['approve', 'reject']}
            ]
        },
        id='1ae34a2ac922842f13e5dd606d41e9d8',
        response_schema=None
    )
]

==== 逐个打印 interrupt 请求 ====


{
    'name': 'get_weather',
    'args': {'city': '北京', 'is_forcast': False},
    'description': "中断啦!\n\nTool: get_weather\nArgs: {'city': '北京', 'is_forcast': False}"
}

{'name': 'get_news', 'args': {}, 'description': '中断啦!\n\nTool: get_news\nArgs: {}'}

{
    'name': 'send_email_tool',
    'args': {'body': '你好啊', 'recipient': '15641685664@qq.com', 'subject': '哈哈哈'},
    'description': '发送邮件中断啦'
}

In [20]:
rprint(response["__interrupt__"][0])
print(type(response["__interrupt__"][0]))

Interrupt(
    value={
        'action_requests': [
            {
                'name': 'get_weather',
                'args': {'city': '北京', 'is_forcast': False},
                'description': "中断啦!\n\nTool: get_weather\nArgs: {'city': '北京', 'is_forcast': False}"
            },
            {'name': 'get_news', 'args': {}, 'description': '中断啦!\n\nTool: get_news\nArgs: {}'},
            {
                'name': 'send_email_tool',
                'args': {'body': '你好啊', 'recipient': '15641685664@qq.com', 'subject': '哈哈哈'},
                'description': '发送邮件中断啦'
            }
        ],
        'review_configs': [
            {'action_name': 'get_weather', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']},
            {'action_name': 'get_news', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']},
            {'action_name': 'send_email_tool', 'allowed_decisions': ['approve', 'reject']}
        ]
    },
    id='1ae34a2ac922842f13e5dd606d41e9d8',
    response_schema=None
)

<class 'langgraph.types.Interrupt'>


In [14]:
action_requests = interrupts[0].value["action_requests"]
for action_request in action_requests:
    rprint(action_request)

{
    'name': 'get_weather',
    'args': {'city': '北京', 'is_forcast': False},
    'description': "中断啦!\n\nTool: get_weather\nArgs: {'city': '北京', 'is_forcast': False}"
}

{'name': 'get_news', 'args': {}, 'description': '中断啦!\n\nTool: get_news\nArgs: {}'}

{
    'name': 'send_email_tool',
    'args': {'body': '你好啊', 'recipient': '15641685664@qq.com', 'subject': '哈哈哈'},
    'description': '发送邮件中断啦'
}

#### 第一次 invoke 返回了什么：`__interrupt__` 与 `Interrupt`

**1. `__interrupt__`：暂停时附带的“待审批单”**

正常情况下，`invoke` 返回的是 Agent 运行结束时的 state（主要是 `messages`）。如果运行中途被暂停，LangGraph 会在返回值里额外加一个 `__interrupt__` 键，列出当前等待人工处理的暂停（源码中是 `{**latest, "__interrupt__": interrupts}`）。

对照上面的输出：

* `messages` 里只有提问和一条带 4 个 `tool_calls` 的 AIMessage，没有任何 ToolMessage，说明工具一个都还没执行（包括无需审批的 `read_email_tool`）。Agent 停在“模型决定调用工具之后、工具执行之前”。
* `__interrupt__` 不属于 state，只在暂停时出现，所以用 `response.get("__interrupt__", [])` 判断有没有暂停。
* 它是一个列表，因为一个图里可能同时有多处暂停（见 2.2.4 的例子）。HITL 会把所有需要审批的调用打包成一次暂停，所以列表里只有 1 个 `Interrupt`，其中有 3 个 `action_requests`；`read_email_tool` 配置为 `False`，自动放行，不在其中。

**2. `langgraph.types.Interrupt`：描述“一次暂停”的对象**

| 属性 | 含义 | 本例 |
|---|---|---|
| `value` | 调用 `interrupt(value)` 时传入的内容。LangGraph 不解析它，原样交给调用方，用来展示给审批人 | HITL 的审批请求：`action_requests`（要审批的调用、参数和描述）和 `review_configs`（每个工具允许的决策） |
| `id` | 这次暂停的唯一编号，每次运行都不同 | 只有一个暂停时用不到；同时有多个暂停时，用它指定回复哪一个（见 2.2.4） |
| `response_schema` | 期望的回复格式（可选，方便前端生成表单） | HITL 没有设置，所以是 `None` |

`value` 来自 HITL 中间件的 `after_model` 钩子（钩子是什么、它怎样被一步步调用，见 2.2.5），关键代码只有一行：

```python
decisions = interrupt(hitl_request)["decisions"]
```

`hitl_request` 就是上面看到的 `value`。`interrupt()` 第一次执行时让运行暂停；恢复执行时，它的返回值就是我们传回去的决策字典（见 2.2.3）。

### 2.2.3 举例过程2：指明工具调用请求决策

恢复执行时要传入一个决策字典，格式由 HITL 中间件规定（源码中用 `TypedDict` 定义：`HITLResponse`、`ApproveDecision`、`EditDecision`、`RejectDecision`、`RespondDecision`）。`TypedDict` 只是类型提示，运行时就是普通字典，所以按格式写字典即可：

```python
{"decisions": [
    {"type": "approve"},                                                 # 原样执行
    {"type": "edit", "edited_action": {"name": "工具名", "args": {...}}},  # 按修改后的参数执行
    {"type": "reject", "message": "拒绝理由（可选）"},                       # 不执行，告诉模型被拒绝了
    {"type": "respond", "message": "代替工具返回的内容"},                    # 不执行，由人直接给出工具结果
]}
```

规则：

* 外层必须是 `{"decisions": [...]}`，因为中间件取的是 `["decisions"]`。
* 决策数量必须和 `action_requests` 的数量一致，否则报 `ValueError`。
* 第 i 个决策对应第 i 个 `action_request`，按**位置**匹配，不看工具名，所以顺序必须一致。
* 决策类型必须在该工具的 `allowed_decisions` 里。例如本例 `send_email_tool` 只允许 approve 和 reject，传 edit 会报 `ValueError`。

In [ ]:
# 如果有中断，说明进入人在环了
# 这么定义来自于HumanInTheLoopMiddleware的源码，关键代码是：        
# decisions = interrupt(hitl_request)["decisions"] 以及
# revised_tool_call, tool_message = self._process_decision
weather_decision = {
    "type": "edit",
    "edited_action": {
        "name": "get_weather",
        "args": {
            "city": "中国上海市",
            "is_forcast": True
        }
    }
}
news_decision = {
    "type": "approve",
}
send_email_decision = {
    "type": "approve"
}
decisions = {
    "decisions": []
}
# 决策的顺序必须和返回的中断请求顺序一致
for action_request in action_requests:
    if action_request["name"] == "get_weather":
        decisions["decisions"].append(weather_decision)
    if action_request["name"] == "get_news":
        decisions["decisions"].append(news_decision)
    if action_request["name"] == "send_email_tool":
        decisions["decisions"].append(send_email_decision)
if interrupts:
    # 审批通过
    resumed_response = agent.invoke(
        Command(resume=decisions),
        config=config, # 必须是同一个 thread_id，指定同一个会话
    )
    print("==== 审批后继续执行 ====")
    for msg in resumed_response["messages"]:
        msg.pretty_print()

>>> 真的执行发送邮件工具了
==== 审批后继续执行 ====
================================ Human Message =================================

请帮我查询今天北京的天气查询今日新闻查看ID为 'sk2131421' 的邮件内容，向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'同时做这四件事
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_UDK2lJdr4DAGkDNY01ZHgBAr)
 Call ID: call_UDK2lJdr4DAGkDNY01ZHgBAr
  Args:
    city: 北京
    is_forcast: False
  get_news (call_gN8c7oZkHc3bFHJF0YBFox6Q)
 Call ID: call_gN8c7oZkHc3bFHJF0YBFox6Q
  Args:
  read_email_tool (call_dTJ0PSzr6sSHePM7aV1BctnB)
 Call ID: call_dTJ0PSzr6sSHePM7aV1BctnB
  Args:
    email_id: sk2131421
  send_email_tool (call_29WAnJco5oOKLHWrWfnNaP7Y)
 Call ID: call_29WAnJco5oOKLHWrWfnNaP7Y
  Args:
    body: 你好啊
    recipient: 15641685664@qq.com
    subject: 哈哈哈
================================= Tool Message =================================
Name: get_weather

Note: a human reviewer replaced this tool call before it ran. The call recorded in your messag

#### 为什么传入决策字典就能继续执行

**第一步：第一次 invoke 暂停并存档。** `interrupt()` 第一次被调用时，会在内部抛出一个特殊异常（`GraphInterrupt`），让运行停下来。LangGraph 把当前 state 和“停在哪个节点”按 `thread_id` 存进 checkpointer，然后让 `invoke` 返回。此时程序里没有任何东西在“等待”，所有信息都在 checkpointer 里。所以 HITL 必须配置 checkpointer，恢复时也必须使用同一个 `thread_id`。

**第二步：第二次 invoke 读档，把被中断的节点从头重新执行。** `agent.invoke(Command(resume=decisions), config)` 让 LangGraph 读出该线程的存档，重新执行停下的节点（`HumanInTheLoopMiddleware.after_model`）。代码再次走到 `interrupt(...)` 时不再暂停，而是直接返回 `resume` 里的值，也就是我们的决策字典。

下面用一个不需要模型的小图验证这一点：

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command


class DemoState(TypedDict):
    answer: str


def ask_node(state: DemoState):
    print("[ask_node] 开始执行")
    ans = interrupt("请给出答复")  # 第一次：暂停；恢复时：返回 resume 的值
    print("[ask_node] interrupt 返回了:", ans)
    return {"answer": ans}


demo_builder = StateGraph(DemoState)
demo_builder.add_node("ask_node", ask_node)
demo_builder.add_edge(START, "ask_node")
demo_graph = demo_builder.compile(checkpointer=InMemorySaver())
demo_config = {"configurable": {"thread_id": "demo"}}

print("第一次 invoke 返回:", demo_graph.invoke({"answer": ""}, demo_config))
print("存档中等待执行的节点:", demo_graph.get_state(demo_config).next)
print("第二次 invoke 返回:", demo_graph.invoke(Command(resume="同意"), demo_config))
# 所以类似的 decision的值也被赋予了interrupt的创建函数的返回值

[ask_node] 开始执行
第一次 invoke 返回: {'answer': '', '__interrupt__': [Interrupt(value='请给出答复', id='6185922f535c7255afd7f59e95316b79', response_schema=None)]}
存档中等待执行的节点: ('ask_node',)
[ask_node] 开始执行
[ask_node] interrupt 返回了: 同意
第二次 invoke 返回: {'answer': '同意'}


`[ask_node] 开始执行` 打印了两次：恢复并不是从暂停的那一行接着运行，而是把整个节点重新执行一遍，只是这一次 `interrupt()` 直接返回了 `resume` 的值。因此，节点中位于 `interrupt()` 之前的代码会执行两次，不要在那里放只能执行一次的操作（例如扣款、发通知）。

**第三步：中间件按决策改写工具调用。** HITL 的 `after_model` 拿到决策字典后，按 2.2.3 的规则逐个处理：approve 和 edit 的调用会继续执行；reject 和 respond 直接生成一条 ToolMessage 代替执行结果；无需审批的调用（本例的 `read_email_tool`）自动保留。之后流程照常往下走：执行工具，再调用模型生成最终回答。

对照 2.2.3 的输出：

* AIMessage 里 `get_weather` 的参数还是 `北京 / False`，ToolMessage 显示的却是“中国上海市今天天气不错，明天下雨”。这是因为 edit 不修改历史中模型原来的调用，只在执行时替换参数。
* ToolMessage 前面多了一段 “Note: a human reviewer replaced this tool call…”，这就是 `edit_notice` 的默认文字，告诉模型实际执行的是人修改过的调用。其中的 `中国上海市` 是“中国上海市”的 Unicode 转义写法。
* 正因为有这段说明，模型在最终回答里指出了天气工具实际查询的是上海，而不是北京。

### 2.2.4 Command 是什么

`Command`（`from langgraph.types import Command`）是 LangGraph 的“控制指令”对象。普通的输入或节点返回值只能表达“state 是什么”，`Command` 还能告诉图“接下来怎么做”。它有四个字段：

| 字段 | 作用 |
|---|---|
| `resume` | 对暂停的回复，也就是 `interrupt()` 恢复时的返回值。可以是单个值，也可以是 `{中断id: 回复}` 字典 |
| `update` | 要写入 state 的更新，作用和节点返回的字典相同 |
| `goto` | 接下来跳到哪个（或哪些）节点 |
| `graph` | 指令发给哪个图：默认是当前图，`Command.PARENT` 表示父图（用于子图） |

它主要用在两个地方。

**1. 作为 `invoke` 的输入：恢复暂停的运行（本节的用法）**

`invoke` 的输入通常是新的 state，例如 `{"messages": [...]}`。用 `Command(resume=...)` 包起来后，LangGraph 才知道这是“对暂停的回复”，于是读档、重新执行被中断的节点，并把 `resume` 的值作为 `interrupt()` 的返回值。使用时注意三点：

* 必须包一层 `Command`。如果直接写 `agent.invoke(decisions, config)`，LangGraph 会把字典当成一次新的输入，从头再运行一轮：重新调用模型，又停在新的审批上，原来那批工具调用始终得不到执行。
* 必须使用同一个 `thread_id`。换了 `thread_id`，LangGraph 找不到原来的存档，原来的暂停不会被处理。
* 只有一个暂停时，`resume` 直接传值即可（本例就是这样）；同时有多个暂停时，必须用 `{中断id: 回复}` 分别回复，否则会报 `RuntimeError`。下面的例子中，两个并行节点同时暂停：

In [ ]:
import operator
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command


class ParallelState(TypedDict):
    answers: Annotated[list[str], operator.add]  # 两个节点的结果追加到同一个列表


def ask_a(state: ParallelState):
    return {"answers": [f"A:{interrupt('A 需要审批')}"]}


def ask_b(state: ParallelState):
    return {"answers": [f"B:{interrupt('B 需要审批')}"]}


parallel_builder = StateGraph(ParallelState)
parallel_builder.add_node("ask_a", ask_a)
parallel_builder.add_node("ask_b", ask_b)
parallel_builder.add_edge(START, "ask_a")
parallel_builder.add_edge(START, "ask_b")  # ask_a 和 ask_b 并行执行
parallel_graph = parallel_builder.compile(checkpointer=InMemorySaver())
parallel_config = {"configurable": {"thread_id": "parallel"}}

parallel_result = parallel_graph.invoke({"answers": []}, parallel_config)
for item in parallel_result["__interrupt__"]:
    print(item.id, item.value)

# 同时有两个暂停：用 {中断id: 回复} 分别回复
resume_map = {
    item.id: ("同意" if item.value.startswith("A") else "拒绝")
    for item in parallel_result["__interrupt__"]
}
print(parallel_graph.invoke(Command(resume=resume_map), parallel_config))

7cab18b1c0dc8b97a282758439876509 A 需要审批
360613145e4be8782ca6609aac68d6c3 B 需要审批
{'answers': ['A:同意', 'B:拒绝']}


**2. 作为节点（或工具）的返回值：同时更新 state 并指定下一步**

节点返回普通字典只能更新 state，下一步去哪里由图中的边决定；返回 `Command(update=..., goto=...)` 则可以一步完成“更新 state + 跳转”。下面的 `node_a` 直接跳到 `node_c`，跳过了 `node_b`：

In [ ]:
import operator
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command


class PathState(TypedDict):
    path: Annotated[list[str], operator.add]  # 记录经过的节点


def node_a(state: PathState):
    return Command(update={"path": ["a"]}, goto="node_c")  # 更新 state，并直接跳到 node_c


def node_b(state: PathState):
    return {"path": ["b"]}


def node_c(state: PathState):
    return {"path": ["c"]}


path_builder = StateGraph(PathState)
path_builder.add_node("node_a", node_a)
path_builder.add_node("node_b", node_b)
path_builder.add_node("node_c", node_c)
path_builder.add_edge(START, "node_a")
path_builder.add_edge("node_b", "node_c")
path_builder.add_edge("node_c", END)
print(path_builder.compile().invoke({"path": []}))

{'path': ['a', 'c']}


`path` 是 `['a', 'c']`，`node_b` 没有执行。LangChain 的工具函数也可以返回 `Command(update=...)`，用来直接修改 Agent 的 state。HITL 中间件只用到了第一种用法：`Command(resume=...)`。

### 2.2.5 源码追踪：从 agent.invoke 到 after_model

前面多次提到“HITL 中间件的 `after_model` 钩子”。但我们的代码里只写了 `agent.invoke(...)`，从没直接调用过 `after_model`，它是怎么被执行的？`__interrupt__` 里的 `value` 又是怎么来的？本节顺着源码把这条链路走一遍。

> 下文的源码文件都在 `.venv/lib/python3.11/site-packages/` 下，行号对应本项目安装的 langchain 1.4.2 和 langgraph 1.2.12，升级后可能变化。

#### 1. 建图：钩子变成图中的节点

`after_model` 是中间件的六个钩子之一，表示“每次模型调用结束后，紧接着执行”（六个钩子见 01 笔记 1.4 的钩子表）。

`create_agent` 返回的 agent 不是普通函数，而是一张 LangGraph 图：`model` 节点负责调用模型，`tools` 节点负责执行工具。建图时，`create_agent` 会检查每个中间件重写了哪些钩子方法。HITL 重写了 `after_model`，于是：

* 把这个方法包装成一个图节点，名字是“中间件名.钩子名”，即 `HumanInTheLoopMiddleware.after_model`（`langchain/agents/factory.py` 第 1599–1621 行，关键是 `graph.add_node(f"{m.name}.after_model", after_node)`，`after_node` 包装的就是 `m.after_model`）；
* 从 `model` 节点连一条边到这个节点（同一文件第 1789 行：`graph.add_edge("model", ...)`）。

所以这个 agent 的图是：

```
__start__ → model → HumanInTheLoopMiddleware.after_model ─┬→ tools → 回到 model
                                                          └→ __end__（模型没有调用工具时）
```

HITL 挂在这里，是因为此时模型已经在 AIMessage 里给出了 `tool_calls`，而工具还没有执行，正好是审批的时机。另外三点：

* 每次模型调用之后都会经过这个节点。本例模型被调用了两次：第一次给出工具调用，第二次生成最终回答；第二次没有 `tool_calls`，`after_model` 直接返回 `None`，什么也不做。
* 多个中间件都有 `after_model` 时，按 `middleware` 列表的**倒序**执行（紧跟在 `model` 后面的是列表里最后一个中间件）。
* edit 时的参数替换和那段 “Note…” 说明，是 HITL 的另一个钩子 `wrap_tool_call` 在执行工具时完成的，不在 `after_model` 里。

#### 2. 运行：引擎沿着边调用节点

`agent.invoke(...)` 启动的是 LangGraph 的执行引擎。引擎沿着边一个节点一个节点地执行：先执行 `model` 节点调用模型，再执行 `HumanInTheLoopMiddleware.after_model` 节点。“执行这个节点”，就是调用 HITL 的 `after_model(state, runtime)` 方法。调用链（简化）如下，第 5 部分的示例会把它实际打印出来：

```
我们的代码：agent.invoke(...)
  → langgraph/pregel/main.py      invoke
  → langgraph/pregel/main.py      stream           逐步执行图
  → langgraph/pregel/_runner.py   tick             执行当前这一步的节点
  → langgraph/pregel/_retry.py    run_with_retry
  → HumanInTheLoopMiddleware.after_model(state, runtime)
```

也就是说，`after_model` 是引擎替我们调用的，在自己的代码里看不到这一步。

#### 3. 第一次 invoke：`interrupt` 的 `value` 从哪来

`after_model` 被调用后，依次做这几件事（`langchain/agents/middleware/human_in_the_loop.py`）：

1. 从 state 中取出最后一条 AIMessage（第 447 行），也就是 `model` 节点刚写入、带 4 个 `tool_calls` 的那条。
2. 遍历 `tool_calls`，为 `interrupt_on` 中需要审批的调用各生成一条 `action_request` 和一条 `review_config`（第 460 行）：
    * `name`、`args` 直接取自模型给出的 tool_call；
    * `description` 默认拼成 `前缀\n\nTool: 工具名\nArgs: 参数`（第 326 行），所以 2.2.2 中看到的是 `"中断啦!\n\nTool: get_weather…"`；单独配置了 description 的工具直接使用配置的值（第 324 行），所以 `send_email_tool` 是 `"发送邮件中断啦"`；
    * `allowed_decisions` 取自 `interrupt_on`，其中 `True` 在创建中间件时就被换成了四种决策（第 292 行）。
3. 打包成 `hitl_request = {"action_requests": [...], "review_configs": [...]}`（第 475 行）。
4. 调用 `interrupt(hitl_request)`（第 481 行）。

`interrupt()` 内部会先检查有没有答复（`langgraph/types.py` 第 1006 行）。第一次没有，于是执行第 1021 行：把传入的 `hitl_request` 装进一个 `Interrupt` 对象，作为 `GraphInterrupt` 异常抛出。异常从 `after_model` 一路传回引擎，引擎捕获它并存档，然后让 `invoke` 返回 `{**state, "__interrupt__": [Interrupt(...)]}`（`langgraph/pregel/main.py` 第 3951 行）。

所以 `response["__interrupt__"][0].value` 就是第 3 步拼出来的 `hitl_request`，原封不动。

#### 4. 恢复执行：再次进入 `after_model`

`agent.invoke(Command(resume=decisions), config)` 走的是同一条调用链。区别在于：引擎先读出该 `thread_id` 的存档，发现上次停在 `HumanInTheLoopMiddleware.after_model`，于是重新执行这个节点。`after_model` 从第一行开始再跑一遍，拼出同样的 `hitl_request`，再次调用 `interrupt(hitl_request)`。这一次 `interrupt()` 找到了答复，直接把我们的决策字典返回（`langgraph/types.py` 第 1006–1019 行），代码继续往下处理决策（见 2.2.3 中的“第三步”）。

#### 5. 动手追踪：把调用链打印出来

下面的示例用一个结构相同但更小的 Agent（只有一个 `trace_weather` 工具），在两个位置“套一层打印”，观察执行顺序：

* 给 HITL 实例的 `after_model` 套一层：每次被引擎调用时打印第几次，第一次还打印调用链；
* 给 `after_model` 里调用的 `interrupt` 套一层：打印传入的 `value`，以及它是抛出了异常还是返回了答复。

为了不调用真实模型、每次输出一致，这里用 LangChain 自带的假模型 `GenericFakeChatModel`：它按顺序返回预先写好的消息（第一次返回一个工具调用，第二次返回最终回答）。示例结束时会把 `interrupt` 恢复成原函数，不影响笔记中的其他单元格。

In [ ]:
import traceback
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage, HumanMessage
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
import langchain.agents.middleware.human_in_the_loop as hitl_module
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.errors import GraphInterrupt
from langgraph.types import Command


class FakeToolModel(GenericFakeChatModel):
    """假模型：按顺序返回预先写好的消息，不调用任何接口。"""

    def bind_tools(self, tools, **kwargs):
        return self  # create_agent 会调用 bind_tools，假模型直接返回自身


@tool
def trace_weather(city: str) -> str:
    """查询指定城市天气"""
    print(f"      >>> trace_weather 真的执行了，city={city}")
    return f"{city}今天晴"


# 1) 给 HITL 实例的 after_model 套一层打印（必须在 create_agent 之前设置）
trace_hitl = HumanInTheLoopMiddleware(interrupt_on={"trace_weather": True})
original_after_model = trace_hitl.after_model
after_model_calls = {"n": 0}


def traced_after_model(state, runtime):
    after_model_calls["n"] += 1
    print(f"    ② 引擎第 {after_model_calls['n']} 次执行 HumanInTheLoopMiddleware.after_model 节点")
    if after_model_calls["n"] == 1:  # 调用链只打印一次
        print("       调用链：")
        for frame in traceback.extract_stack()[:-1]:
            if "langgraph/pregel" in frame.filename:
                print(f"         {frame.filename.split('site-packages/')[-1]}:{frame.lineno}  {frame.name}")
    return original_after_model(state, runtime)


trace_hitl.after_model = traced_after_model

# 2) 给 after_model 里调用的 interrupt 套一层打印
real_interrupt = hitl_module.interrupt


def traced_interrupt(value):
    print("    ③ after_model 调用 interrupt(value)，value 的键:", list(value.keys()))
    try:
        result = real_interrupt(value)
    except GraphInterrupt:
        print("    ④ interrupt 没有找到答复 → 抛出 GraphInterrupt，运行暂停")
        raise
    print("    ④ interrupt 找到了答复 → 直接返回:", result)
    return result


hitl_module.interrupt = traced_interrupt
try:
    trace_model = FakeToolModel(messages=iter([
        AIMessage("", tool_calls=[{"name": "trace_weather", "args": {"city": "北京"}, "id": "call_1"}]),
        AIMessage("北京今天晴"),
    ]))
    trace_agent = create_agent(model=trace_model, tools=[trace_weather],
                               checkpointer=InMemorySaver(), middleware=[trace_hitl])
    trace_config = {"configurable": {"thread_id": "trace"}}
    print("图中的节点:", list(trace_agent.get_graph().nodes))

    print("\n① 第一次 trace_agent.invoke(...)")
    trace_first = trace_agent.invoke({"messages": [HumanMessage("北京天气怎么样")]}, trace_config)
    print("    ⑤ invoke 返回，存档中停在:", trace_agent.get_state(trace_config).next)
    print("       __interrupt__[0].value =", trace_first["__interrupt__"][0].value)

    print("\n① 恢复：trace_agent.invoke(Command(resume=...))")
    trace_second = trace_agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), trace_config)
    print("    ⑤ invoke 返回，最后一条消息:", trace_second["messages"][-1].content)
finally:
    hitl_module.interrupt = real_interrupt  # 恢复原函数，避免影响笔记里的其他单元格

图中的节点: ['__start__', 'model', 'tools', 'HumanInTheLoopMiddleware.after_model', '__end__']

① 第一次 trace_agent.invoke(...)
    ② 引擎第 1 次执行 HumanInTheLoopMiddleware.after_model 节点
       调用链：
         langgraph/pregel/main.py:3913  invoke
         langgraph/pregel/main.py:2967  stream
         langgraph/pregel/_runner.py:207  tick
         langgraph/pregel/_retry.py:617  run_with_retry
    ③ after_model 调用 interrupt(value)，value 的键: ['action_requests', 'review_configs']
    ④ interrupt 没有找到答复 → 抛出 GraphInterrupt，运行暂停
    ⑤ invoke 返回，存档中停在: ('HumanInTheLoopMiddleware.after_model',)
       __interrupt__[0].value = {'action_requests': [{'name': 'trace_weather', 'args': {'city': '北京'}, 'description': "Tool execution requires approval\n\nTool: trace_weather\nArgs: {'city': '北京'}"}], 'review_configs': [{'action_name': 'trace_weather', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}

① 恢复：trace_agent.invoke(Command(resume=...))
    ② 引擎第 2 次执行 HumanInTheLoopMiddleware.after_mode

对照输出：

* 图中确实有一个名为 `HumanInTheLoopMiddleware.after_model` 的节点（第 1 部分）。
* 第一次 invoke：引擎沿着 `invoke → stream → tick → run_with_retry` 调用到 `after_model`（第 2 部分）；`after_model` 把 `hitl_request` 传给 `interrupt`，`interrupt` 没有找到答复，抛出异常暂停；返回值中的 `__interrupt__[0].value` 正是这个 `hitl_request`（第 3 部分）。
* 恢复执行：引擎**第 2 次**执行同一个节点，这次 `interrupt` 直接返回了决策字典，工具随即执行（第 4 部分）。
* **第 3 次**执行发生在模型第二次返回之后：这次没有 `tool_calls`，`after_model` 没有调用 `interrupt`，直接放行，Agent 结束运行。